# Thrifted Jeans third-stage one-sided EMA audit

This is a separated research-only follow-up. It first reproduces the prior corrected-EMA asymmetry sample, then tests a causal rolling one-step EMA residual using only negative weak-state deviations as buy-the-dip signals. Strong negative Kalman states may still short; the EMA branch never shorts.

The primary rolling residual requires all 20 prior residuals, centers by their prior rolling mean, and excludes today’s residual from both mean and scale.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

AUDIT_DIR = Path.cwd()
if AUDIT_DIR.name != 'thrifted_jeans':
    AUDIT_DIR = Path(r'D:/Documents/Algojam/research/thrifted_jeans')
sys.path.insert(0, str(AUDIT_DIR))
from jeans_one_sided_ema_audit import run_audit

OUTPUT_DIR = AUDIT_DIR / 'one_sided_outputs'
FIGURE_DIR = AUDIT_DIR / 'one_sided_figures'

In [2]:
# Reuse a completed run when present; deleting one_sided_outputs/ forces a rebuild.
manifest_path = OUTPUT_DIR / 'one_sided_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    expected = (300, 200, 300, 500)
    observed = (manifest.get('additive_repetitions_per_block'), manifest.get('drift_repetitions_per_assumption'), manifest.get('regime_repetitions_per_scenario'), manifest.get('familywise_repetitions'))
else:
    manifest, observed, expected = {}, (), (300, 200, 300, 500)
if observed == expected:
    RESULT = {
        'comparison': pd.read_csv(OUTPUT_DIR / 'exact_candidate_comparison.csv'),
        'legacy': pd.read_csv(OUTPUT_DIR / 'legacy_asymmetry_reproduction.csv'),
        'buckets': pd.read_csv(OUTPUT_DIR / 'weak_state_residual_buckets.csv'),
        'correctness': pd.read_csv(OUTPUT_DIR / 'correctness_checks.csv'),
        'manifest': manifest,
    }
    print('Loaded the completed one-sided EMA audit run.')
else:
    RESULT = run_audit(
        output_dir=OUTPUT_DIR,
        figure_dir=FIGURE_DIR,
        additive_repetitions=300,
        drift_repetitions=200,
        regime_repetitions=300,
        familywise_repetitions=500,
    )
RESULT['comparison'][['Candidate', 'P&L', 'Incremental vs K2', 'Incremental vs Candidate C', 'Weak P&L', 'Weak dip-entry P&L', 'Max drawdown', 'Turnover']].round(2)

Loaded the completed one-sided EMA audit run.


,Candidate,P&L,Incremental vs K2,Incremental vs Candidate C,Weak P&L,Weak dip-entry P&L,Max drawdown,Turnover
0,A_always_long,37760.0,-62256.0,-81824.0,23960.0,0.0,-44856.0,800
1,B_K2,100016.0,0.0,-19568.0,23960.0,0.0,-26672.0,29600
2,C_candidate_C,119584.0,19568.0,0.0,23960.0,0.0,-35512.0,32800
3,D_corrected_hybrid_frozen,151584.0,51568.0,32000.0,55960.0,0.0,-14536.0,53600
4,one_sided_primary,99528.0,-488.0,-20056.0,3904.0,3904.0,-24696.0,51200
5,one_sided_hysteresis_exit0,102584.0,2568.0,-17000.0,6960.0,3904.0,-28680.0,47200
6,one_sided_hysteresis_exit025,106296.0,6280.0,-13288.0,10672.0,3904.0,-25880.0,45600
7,no_short_long_flat,58616.0,-41400.0,-60968.0,3904.0,3904.0,-13456.0,35200
8,reduced_weak_long_200,104542.0,4526.0,-15042.0,8918.0,3904.0,-27400.0,46600
9,two_sided_stateless,96256.0,-3760.0,-23328.0,632.0,3904.0,-18024.0,70400


In [3]:
comparison = RESULT['comparison'].set_index('Candidate')
legacy = RESULT['legacy'].set_index('Bucket')
assert comparison.loc['A_always_long', 'P&L'] == 37760.0
assert comparison.loc['B_K2', 'P&L'] == 100016.0
assert comparison.loc['C_candidate_C', 'P&L'] == 119584.0
assert comparison.loc['D_corrected_hybrid_frozen', 'P&L'] == 151584.0
assert legacy.loc['negative_deviation_long_reversal', 'Observations'] == 38
assert legacy.loc['positive_deviation_short_reversal', 'Observations'] == 48
assert abs(legacy.loc['negative_deviation_long_reversal', 'Long reversal hit rate'] - (28 / 38)) < 1e-12
assert abs(legacy.loc['positive_deviation_short_reversal', 'Short reversal hit rate'] - (25 / 48)) < 1e-12
assert RESULT['correctness']['Value'].astype(str).str.lower().eq('true').all()
print('Baseline, asymmetry, and correctness checks passed.')
print('CSV count:', len(list(OUTPUT_DIR.glob('*.csv'))), 'figure count:', len(list(FIGURE_DIR.glob('*.png'))))

Baseline, asymmetry, and correctness checks passed.
CSV count: 19 figure count: 6
